# Total return swaps and variance swaps

**Prerequisites:** **05**–**06**. **`trs_equity`** for index TRS vs a **financing leg**; **`variance_swap`** for realized vs strike variance with **`vega`-style** metrics when registered.


## Concept

- **TRS:** receive/pay **total return** (price + dividends) vs **funded financing** (curve + spread).
- **Variance swap:** payoff on realized variance vs **strike variance**; **vega notional** scales sensitivity.

Variance swap PV may be near zero without a full forward variance / surface stack—focus on the metric plumbing.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, ForwardCurve, MarketContext

print("Imports OK (TRS / variance).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

trs = json.dumps(
    {
        "type": "trs_equity",
        "spec": {
            "id": "TRS-SPX-1Y",
            "side": "receive_total_return",
            "notional": {"amount": "5000000", "currency": "USD"},
            "schedule": {
                "start": AS_OF_STR,
                "end": "2026-01-15",
                "params": {
                    "freq": {"count": 3, "unit": "months"},
                    "dc": "Act360",
                    "bdc": "following",
                    "calendar_id": "weekends_only",
                    "end_of_month": False,
                    "payment_lag_days": 0,
                    "stub": "None",
                },
            },
            "underlying": {
                "ticker": "SPX",
                "currency": "USD",
                "contract_size": 1.0,
                "spot_id": "SPX-SPOT",
                "div_yield_id": "SPX-DIV",
            },
            "financing": {
                "discount_curve_id": "USD-OIS",
                "forward_curve_id": "USD-SOFR-3M",
                "spread_bp": "75",
                "day_count": "Act360",
            },
            "dividend_tax_rate": 0.0,
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

var_swap = json.dumps(
    {
        "type": "variance_swap",
        "spec": {
            "id": "VARSPX-1Y",
            "side": "receive",
            "notional": {"amount": "1000000", "currency": "USD"},
            "start_date": AS_OF_STR,
            "maturity": "2026-01-15",
            "strike_variance": 0.04,
            "underlying_ticker": "SPX",
            "observation_freq": {"count": 1, "unit": "days"},
            "realized_var_method": "CloseToClose",
            "day_count": "Act365F",
            "discount_curve_id": "USD-OIS",
            "attributes": {},
            "pricing_overrides": {},
        },
    }
)

for label, js in (("trs_equity", trs), ("variance_swap", var_swap)):
    try:
        validate_instrument_json(js)
        print(label, "JSON OK")
    except Exception as exc:
        print(label, "validate:", type(exc).__name__, exc)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    [(0.0, 0.05), (5.0, 0.045)],
    base_date=AS_OF,
    day_count="act_360",
)
mc = MarketContext().insert(ois).insert(sofr)
state = json.loads(mc.to_json())
state.setdefault("prices", {})
state["prices"]["SPX-SPOT"] = {"price": {"amount": 4500.0, "currency": "USD"}}
state["prices"]["SPX-DIV"] = {"unitless": 0.015}
market_json = json.dumps(state)
print("market ready for TRS + variance")


## Pricing


In [ ]:
for label, js in (("trs", trs), ("variance", var_swap)):
    try:
        out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
        print(label, ValuationResult.from_json(out))
    except Exception as exc:
        print(label, "price:", type(exc).__name__, exc)


## Metrics


In [ ]:
for label, js, mets in (
    ("trs", trs, ["financing_annuity", "index_delta", "equity_value"]),
    ("variance", var_swap, ["vega", "implied_vol", "variance_vega"]),
):
    try:
        out = price_instrument_with_metrics(
            js, market_json, AS_OF_STR, model="discounting", metrics=mets
        )
        vr = ValuationResult.from_json(out)
        print("—", label)
        for m in mets:
            v = vr.get_metric(m)
            if v is not None:
                print(f"  {m}: {v}")
    except Exception as exc:
        print(label, "metrics:", type(exc).__name__, exc)
print("Vega notional scales variance swap sensitivity to implied vol changes (see deal terms).")


## Takeaways

- **TRS** splits **asset leg** vs **financing leg**; curves must include `forward_curve_id` for float financing.
- **Variance swaps** need rich forward variance inputs for non-trivial PV; metrics still demonstrate the hook.
- Always **`validate_instrument_json`** after edits to long JSON specs.
